# Sesión 3 — ¿Podemos confiar en lo que vemos?

**Curso:** Visualización de Datos y Gerencia de la Información  
**Sesión:** 3 de 9  
**Tipo:** Notebook guiado (caso del docente: Golden State Realty Advisors)

La S2 dejó una primera lista de 1 032 «gangas» y tres sospechas: 965 distritos valen exactamente USD 500 001,
1 273 tienen exactamente 52 años y las dos mejores «gangas» valen USD 14 999. Hoy se revisa la calidad de los datos,
se limpia **lo que la pregunta necesita** y se guarda la tabla revisada de la que parten las sesiones siguientes. Este notebook reproduce cada
cifra y cada gráfico de la presentación.

## Agenda de la sesión

| Parte | Contenido |
|---|---|
| Quiz | Quiz de la sesión 2 |
| Repaso | La primera lista y sus sospechas |
| Parte 1 | Qué es un dato de calidad |
| Parte 2 | Valores límite |
| Parte 3 | Valores imposibles y distritos diminutos |
| Parte 4 | Datos faltantes |
| Parte 5 | La bitácora completa y su efecto en la lista |
| Parte 6 | La tabla para las sesiones siguientes |
| Parte 7 | CONECTA: lo que dejó la S2 y lo que cambia hoy |

## Objetivos de aprendizaje

1. Definir la calidad de un dato como aptitud para el uso y reconocer sus cuatro categorías (Wang y Strong, 1996).
2. Detectar valores límite, valores imposibles y grupos demasiado pequeños con gráficos de distribución.
3. Distinguir los mecanismos de los datos faltantes (MCAR, MAR y MNAR) y decidir si hay que tratarlos (van Buuren, 2018, §1.2–1.3).
4. Registrar cada decisión de limpieza en una bitácora y medir su efecto en el resultado.

## Problemática de negocio

El comité de inversión de Golden State decide en qué distritos hace el equipo su debida diligencia (capacidad: 150
por trimestre). En la S2 el equipo sacó 1 032 «gangas» con datos sin revisar. Antes de llevar la lista al comité,
hay que responder: **¿cuáles de esas «gangas» son reales y cuáles son problemas de datos?**

## Preparación del entorno

In [ ]:
import pathlib

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# El notebook busca datos/ e imagenes/ en su carpeta (o en la de arriba, si está en solucionario/).
BASE = pathlib.Path(".") if pathlib.Path("datos").exists() else pathlib.Path("..")
DATOS, IMAGENES = BASE / "datos", BASE / "imagenes"
IMAGENES.mkdir(exist_ok=True)
# Lo que produce el taller se guarda en salidas/, junto al notebook que se ejecuta (el del estudiante en su carpeta;
# el del solucionario, en solucionario/salidas/): así el solucionario no deja respuestas en la carpeta del estudiante.
SALIDAS = pathlib.Path("salidas")
SALIDAS.mkdir(exist_ok=True)

PALETA = {"principal": "#0F766E", "principal_claro": "#4E9F96", "marino": "#1F2A44", "secundario": "#5B6475",
          "neutro": "#8A93A3", "fondo": "#F3F5F9", "acento": "#B85C0A", "texto": "#1F2937"}
plt.rcParams.update({
    "font.family": "Calibri", "font.size": 11, "axes.edgecolor": PALETA["secundario"],
    "axes.labelcolor": PALETA["texto"], "xtick.color": PALETA["secundario"], "ytick.color": PALETA["secundario"],
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 150, "savefig.dpi": 150,
    "savefig.bbox": "tight"})
# Con Calibri, matplotlib no dibuja bien los tamaños enteros 8 y 9 (se usan 8.5, 9.5 o más) ni el texto a 110 ppp: figure.dpi = 150.


def miles(n, redondeo=0):
    """Entero con espacio de miles; redondeo=-3 redondea a miles."""
    return f"{round(n, redondeo):,.0f}".replace(",", " ")


def decimal(x, d=1):
    return f"{x:.{d}f}".replace(".", ",")


def titulo(fig, texto, subtitulo, y=1.04):
    fig.text(0.01, y, texto, fontsize=15, fontweight="bold", color=PALETA["texto"], ha="left")
    fig.text(0.01, y - 0.055, subtitulo, fontsize=11, color=PALETA["secundario"], ha="left")


def fuente(fig, texto):
    fig.text(0.01, -0.04, texto, fontsize=8.5, color=PALETA["secundario"], ha="left")


def guardar(fig, nombre):
    """Guarda la figura en imagenes/ (la presentación usa estos archivos) y la muestra."""
    fig.savefig(IMAGENES / nombre)
    plt.show()


def limpiar_ejes(ax, x=True, y=False):
    if x:
        ax.set_xticks([]); ax.spines["bottom"].set_visible(False)
    if y:
        ax.set_yticks([])
    ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0)

In [ ]:
FUENTE_GS = "Fuente: California Housing, censo de 1990 (Pace y Barry, 1997). Cálculos del curso."
df = pd.read_csv(DATOS / "housing.csv")
ZONAS = {"INLAND": "Interior", "<1H OCEAN": "A menos de 1 h del mar", "NEAR OCEAN": "Cerca del mar",
         "NEAR BAY": "Cerca de la bahía", "ISLAND": "Isla"}
df["zona"] = df["ocean_proximity"].map(ZONAS)
df["personas_hogar"] = df["population"] / df["households"]
df["razon_precio_ingreso"] = df["median_house_value"] / (df["median_income"] * 10_000)
umbral_crudo = df["razon_precio_ingreso"].quantile(0.05)
df["ganga_s2"] = df["razon_precio_ingreso"] <= umbral_crudo
print(f"{miles(len(df))} distritos; {miles(df['ganga_s2'].sum())} «gangas» de la S2 (umbral {decimal(umbral_crudo, 2)})")

## Repaso de la sesión anterior

| Lo que dejó la S2 | Cifra |
|---|---|
| «Gangas» (5 % más bajo de la razón precio/ingreso) | 1 032, el 92 % en el interior |
| Las dos mejores «gangas» | USD 14 999 cada una |
| Distritos en el valor máximo | 965, exactamente USD 500 001 |
| Distritos con la edad máxima | 1 273, exactamente 52 años |

In [ ]:
fig = plt.figure(figsize=(13, 5.6))
ax = fig.add_axes([0.0, 0.0, 0.42, 0.9]); ax.set_axis_off(); ax.set_aspect("equal")
ax.scatter(df["longitude"], df["latitude"], s=1, color=PALETA["neutro"], alpha=0.35)
g_ = df[df["ganga_s2"]]
ax.scatter(g_["longitude"], g_["latitude"], s=5, color=PALETA["principal"])
pct_int = 100 * (g_["zona"] == "Interior").mean()
ax.set_title(f"Dónde: {miles(len(g_))} «gangas» (en verde); el {pct_int:.0f} % en el interior", loc="left", fontsize=12)
s_ = fig.add_axes([0.5, 0.0, 0.5, 0.9]); s_.set_axis_off(); s_.set_xlim(0, 1); s_.set_ylim(0, 1)
s_.set_title("Tres sospechas sobre los datos", loc="left", fontsize=12)
mejores = df[df["ganga_s2"]].nsmallest(2, "razon_precio_ingreso")
SOSPECHAS = [(f"USD {miles(mejores['median_house_value'].iloc[0])}",
              f"valen las dos mejores «gangas» (razón {decimal(mejores['razon_precio_ingreso'].iloc[0], 2)} y "
              f"{decimal(mejores['razon_precio_ingreso'].iloc[1], 2)}): ¿precio real o piso del archivo?"),
             (miles((df["median_house_value"] >= 500_001).sum()), "distritos valen exactamente USD 500 001"),
             (miles((df["housing_median_age"] >= 52).sum()), "distritos tienen exactamente 52 años")]
for i, (cifra, texto) in enumerate(SOSPECHAS):
    s_.text(0, 0.86 - 0.3 * i, cifra, fontsize=28, fontweight="bold", color=PALETA["acento"], va="center")
    s_.text(0, 0.74 - 0.3 * i, texto, fontsize=11, color=PALETA["texto"], va="center")
titulo(fig, f"La S2 dejó {miles(len(g_))} «gangas», casi todas en el interior, y tres sospechas sobre los datos",
       "Golden State con los datos sin revisar: cada punto es un distrito", y=1.02)
fuente(fig, FUENTE_GS)
guardar(fig, "repaso_s2.png")

### Fe de erratas de la S2: mismos datos, otra regla

Un grupo de estudiantes encontró un error en el Sankey de CONECTA (guiado de la S2, Parte 2). **Los datos estaban
bien**: cada cliente que se fue en octubre de 2026 tenía su cancelación registrada. **Falló la regla** con que se
contaron: «sigue quien tiene una fila en octubre». Confiar en lo que vemos exige revisar los datos **y** las reglas
con que los contamos; eso hace esta sesión, y la pregunta del taller es la misma: ¿qué cuenta como una cancelación?

Cada cliente se clasifica por su **última fila**. Hay cuatro casos:

In [ ]:
hs = pd.read_csv(DATOS / "conecta_clientes_mes.csv", dtype={"mes": str}).sort_values("mes")
primera = hs.groupby("cliente_id").head(1).set_index("cliente_id")
ultima = hs.groupby("cliente_id").tail(1).set_index("cliente_id")
seg_cliente = pd.Series(np.where(primera["linea"] == "Hogar", "Hogar",
                                 np.where(primera["modalidad"] == "Prepago", "Móvil prepago", "Móvil pospago")),
                        index=primera.index)
ultimo_mes = ultima["mes"] == hs["mes"].max()                  # ¿su última fila es de octubre de 2026?
cancela = ultima["cancelo"] == 1                               # ¿en su última fila se registró una cancelación?
cambio_plan = ultima["tipo_cancelacion"] == "Cambio de plan"   # el sistema cierra un contrato, pero el cliente sigue

# Regla de la S2: solo mira CUÁNDO fue la última fila. Se fue quien canceló antes de octubre.
sale_s2 = cancela & ~ultimo_mes
# Regla corregida: mira QUÉ dice la última fila. Se fue quien termina con una cancelación que no es cambio de plan.
sale_bien = cancela & ~cambio_plan

casos = pd.DataFrame({
    "su última fila": ["Octubre, sin cancelación", "Antes de octubre, se fue", "Octubre, se fue", "Octubre, cambio de plan"],
    "clientes": [int((ultimo_mes & ~cancela).sum()), int((~ultimo_mes & cancela & ~cambio_plan).sum()),
                 int((ultimo_mes & cancela & ~cambio_plan).sum()), int((ultimo_mes & cambio_plan).sum())],
    "regla de la S2": ["Sigue", "Se fue", "Sigue (error)", "Sigue"],
    "regla corregida": ["Sigue", "Se fue", "Se fue", "Sigue"]}, index=["A", "B", "C", "D"])
display(casos)
print(f"La regla de la S2 contó {miles(sale_s2.sum())} salidas (el caso B); la corregida, {miles(sale_bien.sum())} (B y C).")

La regla de la S2 acierta en A, B y D y falla en C: el último mes mezcla a los que siguen con los que se van ese mismo
mes. **Voluntaria** quiere decir que el cliente se va; **cambio de plan**, que el sistema cerró un contrato, pero el
cliente sigue con otro plan.

In [ ]:
def se_va(sale):
    # Porcentaje de clientes de cada segmento que se va en el año (sin el cliente que sale sin registro).
    valido = sale | (ultimo_mes & ~sale)
    return 100 * sale[valido].groupby(seg_cliente[valido]).mean()


erratas = pd.DataFrame({"Se va en el año (S2)": se_va(sale_s2), "Corregido": se_va(sale_bien)})
display(erratas.round(1))

In [ ]:
# El Sankey de la S2, antes y después, con la misma escala.
from matplotlib.patches import Rectangle

IZQ, DER = ["Hogar", "Móvil pospago", "Móvil prepago"], ["Siguen", "Voluntaria", "Por mora", "Sin recarga"]
COLOR = {"Siguen": PALETA["neutro"], "Voluntaria": PALETA["acento"], "Por mora": PALETA["marino"],
         "Sin recarga": PALETA["principal"]}


def flujo_de(sale):
    valido = sale | (ultimo_mes & ~sale)                 # un cliente sale sin registro de cancelación: queda fuera
    destino = pd.Series(np.where(sale, ultima["tipo_cancelacion"], "Siguen"), index=ultima.index)
    return pd.crosstab(seg_cliente[valido], destino[valido]).reindex(index=IZQ, columns=DER, fill_value=0)


def sankey(ax, flujo, titulo_panel):
    total, hueco = flujo.values.sum(), 0.03 * flujo.values.sum()

    def nodos(tamanos):
        pos, cursor = {}, total + hueco * (len(tamanos) - 1)
        for nombre, n in tamanos.items():
            pos[nombre] = [cursor, cursor - n]; cursor -= n + hueco
        return pos

    pi, pd_ = nodos(flujo.sum(axis=1)), nodos(flujo.sum(axis=0))
    libre_i = {k: v[0] for k, v in pi.items()}; libre_d = {k: v[0] for k, v in pd_.items()}
    xs = np.linspace(0, 1, 100); suave = 3 * xs ** 2 - 2 * xs ** 3
    for s_ in IZQ:
        for d_ in DER:
            n = flujo.loc[s_, d_]
            if n == 0:
                continue
            a0, a1 = libre_i[s_], libre_d[d_]
            ax.fill_between(xs, (a0 - n) + (a1 - a0) * suave, a0 + (a1 - a0) * suave, color=COLOR[d_],
                            alpha=0.45 if d_ == "Siguen" else 0.8, linewidth=0)
            libre_i[s_] -= n; libre_d[d_] -= n
    for nombre, (arr, aba) in pi.items():
        ax.add_patch(Rectangle((-0.03, aba), 0.03, arr - aba, color=PALETA["texto"]))
        ax.text(-0.05, (arr + aba) / 2, nombre, ha="right", va="center", fontsize=10.5)
    for nombre, (arr, aba) in pd_.items():
        ax.add_patch(Rectangle((1, aba), 0.03, arr - aba, color=COLOR[nombre]))
        ax.text(1.05, (arr + aba) / 2, f"{nombre}: {miles(arr - aba)}", va="center", fontsize=10.5, color=PALETA["texto"])
    ax.set_xlim(-0.5, 1.6); ax.set_ylim(0, total * 1.12); ax.set_axis_off()
    ax.set_title(titulo_panel, loc="left", fontsize=12)


fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5.6))
f_s2, f_bien = flujo_de(sale_s2), flujo_de(sale_bien)
sankey(a1, f_s2, f"Antes (S2): {miles(sale_s2.sum())} salidas")
sankey(a2, f_bien, f"Después (corregido): {miles(sale_bien.sum())} salidas")
titulo(fig, f"Mismos datos, otra regla: {miles(sale_bien.sum() - sale_s2.sum())} salidas aparecían como clientes que siguen",
       "Sankey de la S2: segmento de cada cliente del año y cómo termina; el ancho de cada banda es el número de "
       "clientes, con la misma escala en los dos paneles (CONECTA, datos sin revisar)", y=1.02)
fuente(fig, "Fuente: CONECTA S.A., datos sintéticos del curso (nov. 2025 – oct. 2026). Conjuntos paralelos: Wilke (2019), pp. 113–116.")
guardar(fig, "sankey_antes_despues.png")

## Parte 1 — Qué es un dato de calidad

Wang y Strong (1996) preguntaron a los usuarios de datos qué entendían por calidad. Su definición: **datos aptos
para el uso** de quien los consume (sección «Related Research»). El título del artículo, *Beyond accuracy* («más
allá de la exactitud»), resume la idea: lo habitual es revisar solo si el dato es correcto, pero quien usa los datos
juzga más: si sirven para su tarea, si se entienden y si puede acceder a ellos. Agruparon las dimensiones en cuatro
categorías (sección «Summary and Conclusions»):

| Categoría | Dimensiones | En Golden State | En CONECTA |
|---|---|---|---|
| Intrínseca | Exactitud, objetividad, credibilidad, reputación | 4 distritos en USD 14 999, el piso del archivo, 3 de ellos «gangas» (exactitud); 37 con más de 10 personas por hogar (credibilidad) | 1 765 filas duplicadas exactas y una cuenta de prueba (exactitud) |
| Contextual | Valor agregado, relevancia, oportunidad, completitud, cantidad apropiada | 207 nulos de dormitorios, que la pregunta no usa (completitud); 234 distritos con menos de 100 habitantes (cantidad apropiada) | cancelo cuenta el cambio de plan como cancelación (relevancia); el NPS falta en el 78 % de las filas (completitud) |
| De representación | Interpretabilidad, facilidad de comprensión, consistencia, concisión | El ingreso viene en decenas de miles de USD: 3,5 son USD 35 000 (interpretabilidad) | 38 formas de escribir 19 localidades (consistencia) |
| De accesibilidad | Accesibilidad, seguridad de acceso | Archivo público (accesibilidad) y agregado por distrito, sin datos personales (seguridad de acceso) | cliente_id y coordenadas de cada cliente: datos personales (seguridad de acceso, S8) |

**Las 15 dimensiones, en una línea.** Cada una es un grado («el grado en que…»), no una condición de sí o no
(Wang y Strong, 1996, apéndice):

| Categoría | Dimensión | Definición |
|---|---|---|
| Intrínseca | Exactitud | Los datos son correctos, confiables y están certificados como libres de error |
|  | Objetividad | Los datos son imparciales, sin sesgo |
|  | Credibilidad | Los datos se aceptan o se consideran verdaderos, reales y creíbles |
|  | Reputación | Se confía en los datos por su fuente o su contenido |
| Contextual | Valor agregado | Usar los datos trae beneficios y ventajas |
|  | Relevancia | Los datos son aplicables y útiles para la tarea |
|  | Oportunidad | La antigüedad de los datos es apropiada para la tarea |
|  | Completitud | Los datos tienen la amplitud, la profundidad y el alcance suficientes para la tarea |
|  | Cantidad apropiada | La cantidad o el volumen de datos disponibles es apropiado |
| De representación | Interpretabilidad | Los datos están en el lenguaje y las unidades apropiados, y sus definiciones son claras |
|  | Facilidad de comprensión | Los datos son claros, sin ambigüedad, y se entienden con facilidad |
|  | Consistencia | Los datos se presentan siempre en el mismo formato y son compatibles con los anteriores |
|  | Concisión | Los datos se representan de forma compacta sin abrumar: breves, pero completos y al grano |
| De accesibilidad | Accesibilidad | Los datos están disponibles o se obtienen con facilidad y rapidez |
|  | Seguridad de acceso | El acceso a los datos se puede restringir y, por tanto, mantener seguro |

La categoría contextual dice que la calidad **depende de la tarea**. Por eso se limpia lo que la pregunta necesita,
no todo lo que se ve raro.

## Los problemas del caso: cuatro revisiones

Cada revisión se hace en una parte de este notebook y el taller las repite con CONECTA:

| Revisión | En Golden State | En CONECTA | Categoría | Dónde |
|---|---|---|---|---|
| 1. Cuántos valores quedan exactamente en el mínimo y el máximo | Valor en USD 14 999 y 500 001; ingreso en 15; edad en 52 | Latitud cerca de −74: 1 048 filas con coordenadas cambiadas | Intrínseca y de representación | Parte 2 |
| 2. Mirar la distribución | Más de 10 personas por hogar; distritos diminutos | 150 cuentas con pago 0 todo el año | Intrínseca y contextual | Parte 3 |
| 3. Contar los nulos y preguntar si la pregunta usa la variable | 207 nulos de dormitorios | El NPS falta en el 78 % de las filas | Contextual | Parte 4 |
| 4. Leer el diccionario y contar valores distintos | Ingreso en decenas de miles de USD; datos agregados | 38 nombres para 19 localidades; cancelo incluye el cambio de plan | De representación, contextual y de accesibilidad | Parte 1 y taller |

## Parte 2 — Valores límite

Un valor que se repite exactamente en el máximo o en el mínimo de una variable es una señal: el archivo no registra
valores más allá de ese límite. El valor real del distrito no se conoce, solo que está en el límite o más allá.

**La revisión:** para cada una de las 9 variables numéricas, cuántos distritos quedan exactamente en el mínimo y en
el máximo, frente al valor más frecuente de la columna.

In [ ]:
filas = []
for col in df.select_dtypes("number").columns.drop(["personas_hogar", "razon_precio_ingreso"]):
    s = df[col].dropna()
    filas.append({"variable": col, "mínimo": s.min(), "en el mínimo": int((s == s.min()).sum()),
                  "máximo": s.max(), "en el máximo": int((s == s.max()).sum()),
                  "valor más frecuente": s.value_counts().index[0]})
revision = pd.DataFrame(filas).set_index("variable")
revision["¿se acumula en un extremo?"] = np.where(revision[["en el mínimo", "en el máximo"]].max(axis=1) >= 10, "Sí", "No")
display(revision)

Solo tres variables se acumulan en un extremo: el valor de la vivienda, el ingreso y la edad. En el valor y en la
edad, el máximo es además el valor más frecuente de toda la columna; en el ingreso, los decimales 15,0001 y 0,4999
delatan un tope y un piso. Las otras seis tienen uno o dos distritos en cada extremo, que es lo normal.

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(12, 3.8))
paneles = [("median_house_value", 1 / 1000, "Valor de la vivienda (miles de USD)"),
           ("median_income", 10, "Ingreso (miles de USD)"),
           ("housing_median_age", 1, "Edad de la vivienda (años)")]
# El límite de cada variable, con la misma regla que usa la bitácora.
LIMITE = {"median_house_value": (500_001, "en USD 500 001"), "median_income": (15, "en USD 150 000 o más"),
          "housing_median_age": (52, "con 52 años")}
for ax, (col, escala, etiqueta) in zip(ejes, paneles):
    x = df[col] * escala
    n, bordes, barras = ax.hist(x, bins=50, color=PALETA["neutro"])
    barras[-1].set_facecolor(PALETA["acento"])
    valor, texto = LIMITE[col]
    en_limite = int((df[col] >= valor).sum())
    ax.text(0.9, 0.95, f"{miles(en_limite)} {texto}", transform=ax.transAxes, ha="right", va="top",
            fontsize=10.5, color=PALETA["acento"])
    ax.set_xlabel(etiqueta)
ejes[0].set_ylabel("Distritos")
titulo(fig, "Tres variables se acumulan en su valor máximo",
       "Distribución de cada variable; en naranja, el último intervalo", y=1.08)
fuente(fig, FUENTE_GS)
guardar(fig, "valores_limite.png")

**De aquí salen tres reglas** (la bitácora las numera igual hasta el final):

- **Regla 1.** Valor de la vivienda en el tope (USD 500 001): se excluye; su razón no es un dato real.
- **Regla 2.** Valor de la vivienda en el piso (USD 14 999): se excluye, por lo mismo.
- **Regla 3.** Ingreso en el tope (15 o más, es decir, USD 150 000 o más): se excluye. Hay 49 distritos en 15,0001 y
  2 en 15,0; la regla usa «15 o más».

La edad en 52 años **no genera regla**: la razón precio/ingreso no usa la edad.

¿Qué tan graves son estos problemas para la lista de la S2? Las 10 mejores «gangas», una por una:

In [ ]:
REGLAS = {   # número de regla: (condición, por qué es sospechosa)
    1: (lambda f: f["median_house_value"] >= 500_001, lambda f: "valor en el tope del archivo"),
    2: (lambda f: f["median_house_value"] <= 14_999, lambda f: "valor en el piso del archivo"),
    3: (lambda f: f["median_income"] >= 15, lambda f: "ingreso en el tope del archivo"),
    4: (lambda f: f["personas_hogar"] > 10, lambda f: "más de 10 personas por hogar"),
    5: (lambda f: f["population"] < 100, lambda f: f"la mediana sale de {f['households']:.0f} hogares"),
    6: (lambda f: f["ocean_proximity"] == "ISLAND", lambda f: "Isla, sin comparables"),
}
SOSPECHOSA = {1: "median_house_value", 2: "median_house_value", 3: "median_income", 5: "population"}

top = df[df["ganga_s2"]].nsmallest(10, "razon_precio_ingreso").copy()
top["reglas"] = top.apply(lambda f: [n for n, (cond, _) in REGLAS.items() if cond(f)], axis=1)
top["por_que"] = top.apply(lambda f: "; ".join(f"Regla {n}: {REGLAS[n][1](f)}" for n in f["reglas"])
                           or "Sin problema: sigue en la lista", axis=1)
top10_con_problema = int((top["reglas"].str.len() > 0).sum())

fig = plt.figure(figsize=(13, 5.2))
ax = fig.add_axes([0.06, 0.08, 0.26, 0.8])
y = np.arange(len(top))[::-1]
colores = [PALETA["acento"] if r else PALETA["principal"] for r in top["reglas"]]
ax.barh(y, top["razon_precio_ingreso"], color=colores, height=0.68)
for yi, v in zip(y, top["razon_precio_ingreso"]):
    ax.text(v + 0.02, yi, decimal(v, 2), va="center", fontsize=10)
ax.set_yticks(y, [f"{i}.º" for i in range(1, len(top) + 1)]); ax.set_ylim(-0.6, len(top) - 0.4)
ax.set_xlim(0, top["razon_precio_ingreso"].max() * 1.35); ax.set_xlabel("Años de ingreso")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: decimal(v, 1)))
ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0)
# La tabla a la derecha, una fila por barra.
tab = fig.add_axes([0.33, 0.08, 0.66, 0.8], sharey=ax); tab.set_axis_off(); tab.set_xlim(0, 1)
COLS = [(0.0, "Valor (USD)", "median_house_value", lambda v: miles(v)),
        (0.13, "Ingreso (USD)", "median_income", lambda v: miles(10_000 * v)),
        (0.27, "Habitantes", "population", lambda v: miles(v)),
        (0.39, "Por qué es sospechosa", "por_que", str)]
for x, encabezado, col, formato in COLS:
    tab.text(x, len(top) - 0.25, encabezado, fontsize=10.5, fontweight="bold", va="bottom", color=PALETA["texto"])
for yi, (_, f) in zip(y, top.iterrows()):
    marcadas = {SOSPECHOSA[n] for n in f["reglas"] if n in SOSPECHOSA}
    for x, _, col, formato in COLS:
        rojo = col in marcadas or (col == "por_que" and f["reglas"])
        tab.text(x, yi, formato(f[col]), va="center", fontsize=10,
                 color=PALETA["acento"] if rojo else (PALETA["principal"] if col == "por_que" else PALETA["texto"]),
                 fontweight="bold" if col in marcadas else "normal")
titulo(fig, f"{top10_con_problema} de las 10 mejores «gangas» de la S2 son problemas de datos",
       "Las 10 razones precio/ingreso más bajas del estado. Naranja: cumple una regla de exclusión (en negrita, la cifra "
       "que la dispara). Verde: sin problema, sigue en la lista", y=1.0)
fuente(fig, FUENTE_GS)
guardar(fig, "mejores_gangas.png")

La regla 5 (menos de 100 habitantes) se explica en la Parte 3: aparece aquí porque cuatro de las mejores «gangas» la
cumplen.

**La bitácora hasta aquí.** Cada decisión se registra con su porqué, también las de no hacer nada:

In [ ]:
bitacora_gs = pd.DataFrame(columns=["regla", "problema", "decisión", "por qué", "distritos"])


def registrar(regla, problema, decision, por_que, distritos):
    bitacora_gs.loc[len(bitacora_gs)] = [regla, problema, decision, por_que, int(distritos)]


registrar("1", "Valor en el tope (USD 500 001)", "Se excluye", "Su razón no es un dato real", (df["median_house_value"] >= 500_001).sum())
registrar("2", "Valor en el piso (USD 14 999)", "Se excluye", "Su razón no es un dato real", (df["median_house_value"] <= 14_999).sum())
registrar("3", "Ingreso en el tope (15 o más)", "Se excluye", "Su razón no es un dato real", (df["median_income"] >= 15).sum())
registrar("—", "Edad en 52 años", "No se toca", "La razón no usa la edad", (df["housing_median_age"] >= 52).sum())
display(bitacora_gs)

### Punto de control 1 — Valores límite

**¿Hay que excluir los 965 distritos en USD 500 001?**
Depende de la pregunta. Para buscar viviendas baratas, ninguno de ellos es candidato: no cambian la lista. Se
excluyen porque su razón no es un dato real, pero la decisión no cambia ningún candidato.

**¿Y los 1 273 distritos con 52 años?**
La razón precio/ingreso no usa la edad: no se excluyen. No todo problema de calidad importa para la pregunta.

> ¿Qué pasaría con la lista si el mínimo de USD 14 999 se hubiera tratado como un precio real?

## Parte 3 — Valores imposibles y distritos diminutos

Dos problemas más aparecen al mirar las distribuciones (Wilke, 2019, p. 83: los valores atípicos se ven en las cajas):

- **Hogares imposibles:** distritos con más de 10 personas por hogar. No es un hogar plausible (criterio del curso).
- **Distritos diminutos:** con menos de 100 habitantes, la mediana del valor y del ingreso sale de muy pocos hogares
  y no es estable (criterio del curso). Su razón típica es normal, pero está mucho más **dispersa**: por eso cae con
  más frecuencia en los extremos, incluido el 5 % más bajo, donde están las «gangas».

In [ ]:
BANDAS = [0, 100, 500, 1_000, 5_000, np.inf]
NOMBRES = ["Menos de 100", "100 a 500", "500 a 1 000", "1 000 a 5 000", "Más de 5 000"]
tamano = pd.cut(df["population"], BANDAS, labels=NOMBRES, right=False)   # sin agregarla a df
por_tamano = df.groupby(tamano, observed=True).agg(
    distritos=("ganga_s2", "size"), p10=("razon_precio_ingreso", lambda s: s.quantile(0.1)),
    mediana=("razon_precio_ingreso", "median"), p90=("razon_precio_ingreso", lambda s: s.quantile(0.9)),
    pct_gangas=("ganga_s2", lambda s: 100 * s.mean()))
imposibles = df[df["personas_hogar"] > 10]
pct_todos = 100 * df["ganga_s2"].mean()
pct_imposibles = 100 * imposibles["ganga_s2"].mean()
display(por_tamano.round(2))
print(f"Hogares imposibles: {len(imposibles)} distritos; {int(imposibles['ganga_s2'].sum())} son «gangas» "
      f"({decimal(pct_imposibles, 1)} % frente a {decimal(pct_todos, 1)} % en todo el estado)")

fig = plt.figure(figsize=(13, 4.8))
ax = fig.add_axes([0.1, 0.12, 0.5, 0.72])
yy = np.arange(len(por_tamano))[::-1]
for yi, (nombre, f) in zip(yy, por_tamano.iterrows()):
    color = PALETA["acento"] if nombre == "Menos de 100" else PALETA["principal"]
    ax.plot([f["p10"], f["p90"]], [yi, yi], color=color, linewidth=7, solid_capstyle="round", alpha=0.85)
    ax.plot(f["mediana"], yi, "o", color="white", markeredgecolor=PALETA["texto"], markersize=8)
    ax.text(15.6, yi, f"{decimal(f['pct_gangas'], 1)} %", va="center", fontsize=11, color=color, fontweight="bold")
ax.text(15.6, len(por_tamano) - 0.35, "«Gangas»", fontsize=10.5, color=PALETA["secundario"])
ax.set_yticks(yy, [f"{n} hab." for n in por_tamano.index]); ax.set_xlim(0, 15.5); ax.set_ylim(-0.6, len(por_tamano) - 0.3)
ax.set_xlabel("Razón precio/ingreso (años de ingreso)")
f0 = por_tamano.loc["Menos de 100"]
ax.annotate(f"El 80 % de estos distritos está entre {decimal(f0['p10'], 1)} y {decimal(f0['p90'], 1)} años",
            (f0["p90"], len(por_tamano) - 1), (8.2, len(por_tamano) - 1.55), fontsize=10, color=PALETA["acento"],
            arrowprops={"arrowstyle": "-", "color": PALETA["acento"], "linewidth": 0.8})
ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0)
ax.set_title("Con pocos habitantes, la razón se dispersa", loc="left", fontsize=12)
tab = fig.add_axes([0.71, 0.12, 0.28, 0.72]); tab.set_axis_off(); tab.set_xlim(0, 1); tab.set_ylim(0, 1)
tab.text(0, 0.97, "Hogares que no existen", fontsize=12, va="top")
for x, h in zip([0, 0.36, 0.66], ["Habitantes", "Hogares", "Personas\npor hogar"]):
    tab.text(x, 0.8, h, fontsize=10.5, fontweight="bold", va="center")
for i, (_, f) in enumerate(imposibles.nlargest(3, "personas_hogar").iterrows()):
    yv = 0.6 - 0.14 * i
    tab.text(0, yv, miles(f["population"]), fontsize=10.5, va="center")
    tab.text(0.36, yv, miles(f["households"]), fontsize=10.5, va="center")
    tab.text(0.66, yv, miles(f["personas_hogar"]), fontsize=10.5, va="center", color=PALETA["acento"], fontweight="bold")
tab.text(0, 0.12, f"{len(imposibles)} distritos con más de 10 personas por hogar;\n"
                  f"{int(imposibles['ganga_s2'].sum())} son «gangas» ({decimal(pct_imposibles, 1)} % frente a "
                  f"{decimal(pct_todos, 1)} % en todo el estado)", fontsize=10.5, va="center", color=PALETA["secundario"])
veces = [pct_imposibles / pct_todos, por_tamano.loc["Menos de 100", "pct_gangas"] / pct_todos]
titulo(fig, f"Los hogares imposibles y los distritos diminutos tienen {int(min(veces))} a {int(np.ceil(max(veces)))} "
            "veces más «gangas» que el estado",
       "Cada barra va del percentil 10 al 90: el 80 % central de los distritos del grupo (no es un intervalo de "
       "confianza); el punto es la mediana. A la derecha de cada barra, el % de «gangas» del grupo", y=1.02)
fuente(fig, FUENTE_GS)
guardar(fig, "imposibles.png")

**De aquí salen dos reglas más:**

- **Regla 4.** Más de 10 personas por hogar: se excluye; no es un hogar plausible (criterio del curso).
- **Regla 5.** Menos de 100 habitantes: se excluye; la mediana no es estable (criterio del curso).

In [ ]:
registrar("4", "Más de 10 personas por hogar", "Se excluye", "No es un hogar plausible (criterio del curso)", (df["personas_hogar"] > 10).sum())
registrar("5", "Menos de 100 habitantes", "Se excluye", "La mediana no es estable (criterio del curso)", (df["population"] < 100).sum())
display(bitacora_gs)

## Parte 4 — Datos faltantes

Rubin clasificó los datos faltantes en tres mecanismos (van Buuren, 2018, §1.2). La pregunta de fondo es **¿de qué
depende que un dato falte?**

| Mecanismo | De qué depende que falte | En palabras | Ejemplo de van Buuren: una báscula | Con el NPS de CONECTA |
|---|---|---|---|---|
| MCAR (completamente al azar) | De nada | Cualquier dato tiene la misma probabilidad de faltar | Se queda sin baterías: falla con cualquier objeto | Las encuestas se pierden por una falla del sistema |
| MAR (al azar) | De algo que **sí está** en los datos | Falta más en algunos grupos, pero los grupos se ven en la tabla | Falla más sobre superficie blanda, y se anota la superficie | Responden menos los clientes con más quejas, y las quejas están en la tabla |
| MNAR (no al azar) | Del **propio valor que falta** | Falta justo por el valor que tendría | Falla más con los objetos más pesados | Los insatisfechos, que darían un NPS bajo, no responden |

El recorrido de esta parte: (1) qué le pasa al promedio con cada mecanismo, (2) cómo saber cuál es, con los
dormitorios de Golden State y el NPS de CONECTA, y (3) qué hacer. La simulación siguiente usa 10 000 clientes
simulados (no son los de CONECTA) cuyo NPS real sí conocemos.

In [ ]:
rng = np.random.default_rng(2026)                        # semilla fija: la simulación siempre da lo mismo
n = 10_000
quejas = rng.choice([0, 1, 2, 3], size=n, p=[0.55, 0.25, 0.12, 0.08])
nps_real = np.clip(np.round(8 - 1.2 * quejas + rng.normal(0, 1.6, n)), 0, 10)
responde = {
    "MCAR: falta por azar": rng.random(n) < 0.25,
    "MAR: falta más con más quejas": rng.random(n) < 0.36 - 0.09 * quejas,
    "MNAR: falta más con un NPS bajo": rng.random(n) < 0.02 + 0.035 * nps_real,
}
PIE = {"MCAR: falta por azar": "Sin sesgo: el promedio de quienes\nresponden es el real",
       "MAR: falta más con más quejas": "Sesgo que se detecta: la respuesta\ncambia con las quejas, que sí vemos",
       "MNAR: falta más con un NPS bajo": "Sesgo que no se detecta: la causa\nes el valor que falta"}
fig, ejes = plt.subplots(1, 3, figsize=(13, 4.2), sharey=True)
bordes = np.arange(-0.5, 11, 1)
for ax, (nombre, r) in zip(ejes, responde.items()):
    ax.hist(nps_real, bins=bordes, density=True, color=PALETA["neutro"], alpha=0.55, label="Todos (real)")
    ax.hist(nps_real[r], bins=bordes, density=True, histtype="step", linewidth=2.2, color=PALETA["principal"],
            label="Quienes responden")
    real, obs = nps_real.mean(), nps_real[r].mean()
    ax.axvline(real, color=PALETA["texto"], linestyle="--", linewidth=1.3)
    ax.axvline(obs, color=PALETA["principal"], linewidth=2)
    ax.set_title(nombre, loc="left", fontsize=11.5)
    ax.text(0.02, 0.97, f"Promedio real {decimal(real, 1)}\nDe quienes responden {decimal(obs, 1)}",
            transform=ax.transAxes, va="top", fontsize=10.5)
    ax.text(0, -0.24, PIE[nombre], transform=ax.transAxes, fontsize=10.5, color=PALETA["secundario"], va="top")
    ax.set_xlabel("NPS (0 a 10)"); ax.set_xticks(range(0, 11, 2))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{100 * v:.0f} %"))
ejes[0].set_ylabel("Clientes")
fig.legend(*ejes[0].get_legend_handles_labels(), frameon=False, fontsize=10.5, loc="upper right", ncol=2, bbox_to_anchor=(0.99, 1.0))
titulo(fig, "Solo cuando falta al azar el promedio de los que responden es el real",
       "Simulación del curso: 10 000 clientes con NPS conocido; cada panel cambia de qué depende que el NPS falte", y=1.06)
fuente(fig, "Fuente: simulación del curso (no son datos de CONECTA). Mecanismos: van Buuren (2018), §1.2.")
guardar(fig, "mecanismos.png")

### Cómo saber cuál es: comparar a quienes tienen el dato con quienes no

En Golden State solo una variable tiene faltantes: `total_bedrooms`. La pregunta: **¿la falta depende de algo que
vemos?** Si los distritos sin dato fueran distintos (más pobres o más caros), no sería MCAR.

In [ ]:
nulo = df["total_bedrooms"].isna()
nulos_candidatos = int((nulo & df["ganga_s2"]).sum())
print(f"total_bedrooms tiene {nulo.sum()} nulos; {nulos_candidatos} entre las «gangas» de la S2")
comparacion = pd.DataFrame({
    "sin dato": df.loc[nulo, ["median_income", "median_house_value", "total_rooms", "razon_precio_ingreso"]].median(),
    "con dato": df.loc[~nulo, ["median_income", "median_house_value", "total_rooms", "razon_precio_ingreso"]].median()})
display(comparacion.round(2))
nivel = pd.qcut(df["median_income"], 3, labels=["Bajo", "Medio", "Alto"])
sin_dato_zona = 100 * nulo[df["zona"] != "Isla"].groupby(df["zona"]).mean().dropna().sort_values()
sin_dato_ingreso = 100 * nulo.groupby(nivel, observed=True).mean()
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8), gridspec_kw={"width_ratios": [1.3, 1]})
fig.subplots_adjust(wspace=0.6)
for ax, serie, titulo_panel in ((a1, sin_dato_zona, "Por zona (sin la Isla: 5 distritos)"),
                                (a2, sin_dato_ingreso, "Por nivel de ingreso del distrito")):
    ax.barh(list(serie.index.astype(str)), serie.values, color=PALETA["principal"], height=0.6)
    for i, v in enumerate(serie.values):
        ax.text(v + 0.02, i, f"{decimal(v, 2)} %", va="center", fontsize=10.5)
    limpiar_ejes(ax); ax.set_xlim(0, 1.6); ax.set_title(titulo_panel, loc="left", fontsize=11.5)
titulo(fig, "La proporción sin dato de dormitorios es pareja en todos los grupos",
       "Distritos sin total_bedrooms ÷ distritos del grupo, en %: si cambiara entre grupos, la falta dependería de algo que vemos",
       y=1.1)
fuente(fig, FUENTE_GS)
guardar(fig, "dormitorios_mcar.png")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
fig.subplots_adjust(wspace=0.08)
for ax, (col, escala, etiqueta) in zip((a1, a2), [("median_income", 10, "Ingreso (miles de USD)"),
                                                  ("median_house_value", 1 / 1000, "Valor (miles de USD)")]):
    ax.boxplot([df.loc[~nulo, col] * escala, df.loc[nulo, col] * escala], orientation="horizontal", patch_artist=True,
               tick_labels=[f"Con dato ({miles((~nulo).sum())})", f"Sin dato ({miles(nulo.sum())})"],
               boxprops={"facecolor": PALETA["principal_claro"], "color": PALETA["principal"]},
               medianprops={"color": PALETA["marino"], "linewidth": 1.5}, flierprops={"markersize": 2})
    ax.set_xlabel(etiqueta)
titulo(fig, "Los distritos sin dato de dormitorios se parecen a los demás",
       "Distribución del ingreso y del valor según si falta total_bedrooms: compatible con MCAR", y=1.08)
fuente(fig, FUENTE_GS)
guardar(fig, "faltantes.png")

Se parecen en ingreso, valor, cuartos y razón, y la proporción sin dato es pareja entre zonas: es **compatible con
MCAR**. No queda probado: si la falta dependiera de algo no observado, los datos no lo mostrarían (van Buuren, 2018,
§1.2).

### El mismo método en CONECTA: el NPS no falta al azar

Para aplicarlo al NPS, se quitan primero las filas que no son clientes (duplicadas, la cuenta de prueba y las cuentas
internas con pago 0 todo el año; las mismas reglas del taller) y se compara la respuesta según las quejas del mes.

In [ ]:
conecta = hs.drop_duplicates()
conecta = conecta[conecta["cliente_id"] != "CNX-PRUEBA-01"]
pago_maximo = conecta.groupby("cliente_id")["ingreso_mensual_cop"].max()
conecta = conecta[~conecta["cliente_id"].isin(pago_maximo[pago_maximo == 0].index)].copy()
respuesta_nps = 100 * conecta.assign(responde=conecta["nps"].notna(), quejas=conecta["quejas_mes"].clip(upper=3)) \
    .groupby("quejas")["responde"].mean()
print(f"{miles(len(conecta))} filas de clientes; el NPS falta en el {decimal(100 * conecta['nps'].isna().mean(), 1)} %")
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.bar(["0", "1", "2", "3 o más"], respuesta_nps.values, color=PALETA["principal"], width=0.6)
for i, v in enumerate(respuesta_nps.values):
    ax.text(i, v + 0.4, f"{decimal(v, 1)} %", ha="center", fontsize=11)
ax.spines["left"].set_visible(False); ax.set_yticks([]); ax.set_ylim(0, respuesta_nps.max() * 1.2)
ax.set_xlabel("Quejas del cliente en el mes")
titulo(fig, "Quien más se queja menos responde la encuesta", "Porcentaje de filas con NPS según las quejas del mes")
fuente(fig, "Fuente: CONECTA S.A., datos sintéticos del curso (nov. 2025 – oct. 2026).")
guardar(fig, "nps_respuesta.png")

Si el NPS faltara completamente al azar, la respuesta sería pareja. Baja con las quejas: **no es MCAR**; depende al
menos de las quejas (MAR) y podría depender de la propia satisfacción (MNAR), lo que los datos no pueden mostrar.
Consecuencia: el NPS promedio de los que responden sobrestima la satisfacción.

### Qué hacer: tres preguntas en orden

Eliminar los casos incompletos solo es insesgado si los datos son MCAR (§1.3.1); los métodos modernos, como la
imputación múltiple, parten del supuesto MAR (§1.2); con MNAR, la estrategia es buscar más datos sobre las causas o
hacer análisis de sensibilidad (§1.2); imputar la media distorsiona la distribución (§1.3.3). **Imputar** es
rellenar el dato que falta con un valor estimado; la **imputación múltiple** lo rellena varias veces, con valores
distintos, y combina los resultados (§1.4).

In [ ]:
from matplotlib.patches import FancyBboxPatch


def caja(ax, x, y, w, h, texto, color, negrita=False):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.012", facecolor="white", edgecolor=color, linewidth=1.8))
    ax.text(x + w / 2, y + h / 2, texto, ha="center", va="center", fontsize=10.5, wrap=True,
            fontweight="bold" if negrita else "normal", color=PALETA["texto"])


def flecha(ax, x0, y0, x1, y1, rotulo=""):
    ax.annotate("", (x1, y1), (x0, y0), arrowprops={"arrowstyle": "-|>", "color": PALETA["secundario"], "linewidth": 1.3})
    if rotulo:
        ax.text((x0 + x1) / 2 + 0.01, (y0 + y1) / 2, rotulo, fontsize=10.5, color=PALETA["secundario"], va="center")


fig, ax = plt.subplots(figsize=(13, 6.2)); ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_axis_off()
caja(ax, 0.02, 0.8, 0.3, 0.13, "1. ¿La pregunta usa\nesa variable?", PALETA["marino"], True)
caja(ax, 0.42, 0.8, 0.56, 0.13, "No se toca. Golden State: la razón no usa los dormitorios.\n"
                                 "CONECTA: la tasa de cancelación no usa el NPS.", PALETA["principal"])
flecha(ax, 0.33, 0.865, 0.42, 0.865)
ax.text(0.375, 0.885, "No", ha="center", fontsize=10.5, color=PALETA["secundario"])
caja(ax, 0.02, 0.5, 0.3, 0.13, "2. ¿Por qué falta?\n(el mecanismo)", PALETA["marino"], True)
flecha(ax, 0.17, 0.8, 0.17, 0.63, "Sí")
for i, (mec, accion) in enumerate([("MCAR", "Eliminar las filas incompletas\nno sesga el promedio"),
                                   ("MAR", "Eliminar sesga; la imputación\nmúltiple parte de este supuesto"),
                                   ("MNAR", "Ningún arreglo con los datos lo corrige:\nlimitación, más datos o escenarios")]):
    yb = 0.6 - 0.2 * i
    caja(ax, 0.42, yb - 0.05, 0.56, 0.13, f"{mec}: {accion}", PALETA["acento"] if mec == "MNAR" else PALETA["principal"])
    flecha(ax, 0.32, 0.565, 0.42, yb + 0.015)
caja(ax, 0.02, 0.02, 0.96, 0.1, "3. Lo que no se hace: rellenar con el promedio. Todos los faltantes quedan con el mismo valor "
                                 "y la distribución se angosta.", PALETA["secundario"])
titulo(fig, "Qué hacer con un dato que falta: tres preguntas en orden",
       "La primera decide casi siempre; las otras dos solo si la pregunta usa la variable", y=1.0)
fuente(fig, "Fuente: van Buuren (2018), §1.2, §1.3.1, §1.3.3 y §1.4.")
guardar(fig, "ruta_faltantes.png")

**Decisión.** Los 207 nulos de `total_bedrooms` se conservan: la razón precio/ingreso no usa esa variable, y la falta
es compatible con MCAR. Si una sesión posterior usara los dormitorios, la decisión se revisaría.

In [ ]:
registrar("—", "Nulos de dormitorios (total_bedrooms)", "No se toca", "La razón no los usa; compatible con MCAR", nulo.sum())
display(bitacora_gs)

### Punto de control 2 — Datos faltantes

**¿Por qué no se eliminan las 12 «gangas» que no tienen dato de dormitorios?**
Porque el indicador no lo usa: eliminarlas perdería candidatos sin ninguna razón.

**¿Se puede demostrar que los faltantes son MCAR?**
No. Con los datos se puede ver que no dependen del ingreso ni del valor; si dependieran de algo no observado
(MNAR), los datos no lo mostrarían (van Buuren, 2018, §1.2).

> ¿Qué faltante sí obligaría a actuar en este caso?

## Parte 5 — La bitácora completa y su efecto en la lista

La bitácora se fue construyendo en las Partes 2 a 4. Falta la **regla 6**: la Isla se excluye porque, con 5
distritos, no tiene comparables (S2 y S4). Con todas las reglas aplicadas se mide el **efecto**: cuántas «gangas» de
la S2 saca cada una.

In [ ]:
registrar("6", "Isla (5 distritos)", "Se excluye", "Sin comparables (S2)", (df["ocean_proximity"] == "ISLAND").sum())
reglas = {
    "1. Valor en el tope (USD 500 001)": df["median_house_value"] >= 500_001,
    "2. Valor en el piso (USD 14 999)": df["median_house_value"] <= 14_999,
    "3. Ingreso en el tope (15 o más)": df["median_income"] >= 15,
    "4. Más de 10 personas por hogar": df["personas_hogar"] > 10,
    "5. Menos de 100 habitantes": df["population"] < 100,
    "6. Isla (5 distritos)": df["ocean_proximity"] == "ISLAND",
}
bitacora = pd.DataFrame({"distritos": {k: int(m.sum()) for k, m in reglas.items()},
                         "gangas_s2": {k: int((m & df["ganga_s2"]).sum()) for k, m in reglas.items()}})
bitacora["decision"] = "Se excluye"
excluir = pd.concat(reglas.values(), axis=1).any(axis=1)
efecto = dict(zip(["1", "2", "3", "4", "5", "6"], bitacora["gangas_s2"]))
bitacora_gs["«gangas» de la S2 que saca"] = bitacora_gs["regla"].map(efecto).map(lambda v: "—" if pd.isna(v) else int(v))
display(bitacora_gs)
print(f"Excluidos: {miles(excluir.sum())} distritos; {int((excluir & df['ganga_s2']).sum())} de las «gangas» de la S2")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.9), sharey=True)
orden = bitacora.index[::-1]
a1.barh(list(orden), bitacora.loc[orden, "distritos"], color=PALETA["neutro"], height=0.6)
a2.barh(list(orden), bitacora.loc[orden, "gangas_s2"],
        color=[PALETA["acento"] if v > 0 else PALETA["neutro"] for v in bitacora.loc[orden, "gangas_s2"]], height=0.6)
for ax, col in ((a1, "distritos"), (a2, "gangas_s2")):
    for regla in orden:
        ax.text(bitacora.loc[regla, col] + bitacora[col].max() * 0.02, regla, miles(bitacora.loc[regla, col]),
                va="center", fontsize=10.5)
    limpiar_ejes(ax); ax.set_xlim(0, bitacora[col].max() * 1.2)
a1.set_title("Distritos excluidos", loc="left", fontsize=11.5)
a2.set_title("«Gangas» de la S2 que elimina", loc="left", fontsize=11.5)
titulo(fig, "La regla que más distritos excluye no cambia ninguna «ganga»",
       "Bitácora de limpieza: el tope de USD 500 001 excluye 965 distritos y ningún candidato", y=1.1)
fuente(fig, FUENTE_GS)
guardar(fig, "bitacora.png")

In [ ]:
limpio = df[~excluir].copy()
umbral = limpio["razon_precio_ingreso"].quantile(0.05)          # el umbral se recalcula con los datos revisados
limpio["candidato"] = limpio["razon_precio_ingreso"] <= umbral
candidatos = limpio[limpio["candidato"]]
pct_interior = 100 * (candidatos["zona"] == "Interior").mean()
print(f"Distritos revisados: {miles(len(limpio))}; umbral: {decimal(umbral, 2)}; candidatos: {miles(len(candidatos))} "
      f"({pct_interior:.0f} % en el interior)")
antes = df[df["ganga_s2"]]["zona"].value_counts()
despues = candidatos["zona"].value_counts()
comp = pd.DataFrame({"S2 (datos sin revisar)": antes, "S3 (datos revisados)": despues}).fillna(0).astype(int)
comp = comp.sort_values("S2 (datos sin revisar)")
fig, ax = plt.subplots(figsize=(9.5, 3.6))
yy = np.arange(len(comp))
ax.barh(yy + 0.2, comp.iloc[:, 0], height=0.38, color=PALETA["neutro"], label=comp.columns[0])
ax.barh(yy - 0.2, comp.iloc[:, 1], height=0.38, color=PALETA["principal"], label=comp.columns[1])
for i, (a, b) in enumerate(comp.to_numpy()):
    ax.text(a + 8, i + 0.2, miles(a), va="center", fontsize=10.5)
    ax.text(b + 8, i - 0.2, miles(b), va="center", fontsize=10.5)
ax.set_yticks(yy, list(comp.index)); limpiar_ejes(ax); ax.set_xlim(0, comp.max().max() * 1.15)
ax.legend(frameon=False, loc="lower right", fontsize=10.5)
titulo(fig, f"Con datos revisados quedan {miles(len(candidatos))} candidatos; el {pct_interior:.0f} % sigue en el interior",
       f"Candidatos por zona antes y después de la limpieza (umbral recalculado: razón ≤ {decimal(umbral, 2)})")
fuente(fig, FUENTE_GS)
guardar(fig, "antes_despues.png")

**La tensión que pasa a la S4.** Los datos ya son confiables, pero la lista no cambió de fondo: 971 candidatos para
una capacidad de 150, casi todos en el interior. ¿Son oportunidades o solo zonas que siempre son más baratas?

## Parte 6 — La tabla para las sesiones siguientes

La S4 reparte los 150 cupos del trimestre por región, y la región no viene en el censo: sale del condado donde está
cada distrito. Se le asigna a cada distrito revisado su condado (el polígono más cercano) y la región del curso, y se
guarda la tabla: es el punto de partida de la S5 y la S6.

In [ ]:
REGIONES = {
    "Bahía de San Francisco": ["Alameda", "Contra Costa", "Santa Clara", "San Francisco", "San Mateo", "Marin",
                               "Sonoma", "Napa", "Solano"],
    "Sur de California": ["Los Angeles", "Orange", "San Diego", "Imperial", "Riverside", "San Bernardino", "Ventura"],
    "Valle Central": ["Kern", "Fresno", "Tulare", "Kings", "Madera", "Merced", "Stanislaus", "San Joaquin",
                      "Sacramento", "Yolo", "Sutter", "Yuba", "Butte", "Glenn", "Colusa", "Tehama", "Shasta"],
}
condados = gpd.read_file(DATOS / "cb_2018_us_county_20m.zip")
condados = condados[condados["STATEFP"] == "06"][["NAME", "geometry"]].rename(columns={"NAME": "condado"}).to_crs(3310)
puntos = gpd.GeoDataFrame(limpio, geometry=gpd.points_from_xy(limpio["longitude"], limpio["latitude"]),
                          crs="EPSG:4326").to_crs(3310)
con_condado = gpd.sjoin_nearest(puntos, condados, how="left")
con_condado = con_condado[~con_condado.index.duplicated()]

distritos = pd.DataFrame(con_condado.drop(columns=["geometry", "index_right", "zona"])).reset_index(names="id_distrito")
region = {c: r for r, lista in REGIONES.items() for c in lista}
distritos["region"] = distritos["condado"].map(region).fillna("Resto del estado")
distritos.to_csv(DATOS / "gs_distritos_s3.csv", index=False, encoding="utf-8")
print(f"{miles(len(distritos))} distritos revisados, cada uno con su condado y su región")
display(distritos[distritos["candidato"]].groupby("region").size().sort_values(ascending=False).to_frame("candidatos"))

## Parte 7 — CONECTA: lo que dejó la S2 y lo que cambia hoy

El taller aplica a CONECTA lo que este notebook hizo con Golden State. Primero, lo que dejó la S2 con los datos sin
revisar; después, lo que cambia al revisarlos y al contar solo las pérdidas (el cambio de plan no es una pérdida).

In [ ]:
upz = gpd.read_file(DATOS / "upz.geojson")
oficiales = set(upz["localidad"])
crudo_oficiales = hs[hs["localidad"].isin(oficiales)]


def segmento(t):
    return np.where(t["linea"] == "Hogar", "Hogar", np.where(t["modalidad"] == "Prepago", "Móvil prepago", "Móvil pospago"))


tasa_s2 = 100 * crudo_oficiales["cancelo"].mean()
por_localidad = 100 * crudo_oficiales.groupby("localidad")["cancelo"].mean().sort_values()
por_segmento_s2 = 100 * hs.assign(segmento=segmento(hs)).groupby("segmento")["cancelo"].mean()
sospechas = [(hs["localidad"].nunique(), f"nombres para {len(oficiales)} localidades"),
             (int((hs["ingreso_mensual_cop"] == 0).sum()), "filas con pago 0"),
             (int((hs["latitud"] < -60).sum()), "filas con latitud y longitud cambiadas")]
fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(13, 5.2), gridspec_kw={"width_ratios": [1.5, 1, 0.9]})
fig.subplots_adjust(wspace=0.45)
a1.barh(list(por_localidad.index.str.title()), por_localidad.values, height=0.7,
        color=[PALETA["principal"] if v == por_localidad.max() else PALETA["neutro"] for v in por_localidad.values])
for i, v in enumerate(por_localidad.values):
    a1.text(v + 0.03, i, decimal(v, 2), va="center", fontsize=9.5)
limpiar_ejes(a1); a1.set_xlim(0, por_localidad.max() * 1.2); a1.tick_params(axis="y", labelsize=9.5)
a1.set_title("Dónde: tasa por localidad (%)", loc="left", fontsize=11.5)
orden = ["Hogar", "Móvil pospago", "Móvil prepago"]
a2.bar([o.replace(" ", "\n") for o in orden], por_segmento_s2[orden].values, width=0.6,
       color=[PALETA["neutro"], PALETA["neutro"], PALETA["principal"]])
for i, v in enumerate(por_segmento_s2[orden].values):
    a2.text(i, v + 0.05, decimal(v, 2), ha="center", fontsize=10.5)
a2.spines["left"].set_visible(False); a2.set_yticks([]); a2.set_ylim(0, por_segmento_s2.max() * 1.25)
a2.tick_params(axis="x", labelsize=9.5); a2.set_title("Quién: tasa por segmento (%)", loc="left", fontsize=11.5)
a3.set_axis_off(); a3.set_title("Tres sospechas", loc="left", fontsize=11.5)
for i, (n, texto) in enumerate(sospechas):
    a3.text(0, 0.85 - 0.33 * i, miles(n), fontsize=26, fontweight="bold", color=PALETA["acento"], va="center")
    a3.text(0, 0.72 - 0.33 * i, texto, fontsize=10.5, color=PALETA["texto"], va="center")
titulo(fig, f"La S2 dejó una tasa de {decimal(tasa_s2, 1)} %, el prepago arriba y tres sospechas sobre los datos",
       "Tasa de cancelación mensual con los datos sin revisar (cancelo), por localidad oficial y por segmento", y=1.04)
fuente(fig, "Fuente: CONECTA S.A., datos sintéticos del curso (nov. 2025 – oct. 2026). Datos sin revisar (S2).")
guardar(fig, "s2_conecta.png")

In [ ]:
# Lo que cambia hoy: los datos revisados (la tabla conecta de la Parte 4) y la pérdida sin cambios de plan.
conecta["perdida"] = ((conecta["cancelo"] == 1) & (conecta["tipo_cancelacion"] != "Cambio de plan")).astype(int)
conecta["segmento"] = segmento(conecta)
perdida_segmento = 100 * conecta.groupby("segmento")["perdida"].mean()
comparar = pd.DataFrame({"S2 (cancelo, sin revisar)": por_segmento_s2, "S3 (perdida, revisados)": perdida_segmento})
comparar.loc["Total"] = [100 * hs["cancelo"].mean(), 100 * conecta["perdida"].mean()]
por_upz = conecta.groupby("codigo_upz").agg(clientes=("cliente_id", "nunique"), tasa_pct=("perdida", lambda s: 100 * s.mean()))
por_upz = por_upz[por_upz["clientes"] >= 80]
candidatas = por_upz.nlargest(round(0.10 * len(por_upz)), "tasa_pct").join(upz.set_index("codigo_upz")[["upz", "localidad"]])
display(comparar.round(2))

fig = plt.figure(figsize=(13, 4.8))
ax = fig.add_axes([0.12, 0.12, 0.5, 0.72])
filas = ["Hogar", "Móvil pospago", "Móvil prepago", "Total"]
for yi, fila in zip(range(len(filas))[::-1], filas):
    a, b = comparar.loc[fila]
    ax.plot([a, b], [yi, yi], color=PALETA["neutro"], linewidth=2.5, zorder=1)
    ax.scatter([a], [yi], s=90, color=PALETA["neutro"], zorder=2)
    ax.scatter([b], [yi], s=90, color=PALETA["principal"], zorder=3)
    ax.text(a + (0.08 if a >= b else -0.08), yi + 0.22, decimal(a, 1), ha="center", fontsize=10.5, color=PALETA["secundario"])
    ax.text(b, yi - 0.38, decimal(b, 1), ha="center", fontsize=10.5, color=PALETA["principal"], fontweight="bold")
ax.set_yticks(range(len(filas))[::-1], filas); ax.set_xlim(0, comparar.max().max() * 1.2); ax.set_ylim(-0.7, len(filas) - 0.4)
ax.set_xlabel("Tasa mensual (%)"); ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: decimal(v, 1)))
ax.scatter([], [], color=PALETA["neutro"], label="S2: cancelo, datos sin revisar")
ax.scatter([], [], color=PALETA["principal"], label="S3: perdida, datos revisados")
ax.legend(frameon=False, fontsize=10, loc="upper right", bbox_to_anchor=(1.0, 1.18), ncol=2)
lista = fig.add_axes([0.68, 0.12, 0.31, 0.72]); lista.set_axis_off(); lista.set_xlim(0, 1); lista.set_ylim(0, 1)
lista.text(0, 1.0, f"Las {len(candidatas)} UPZ candidatas para la S4", fontsize=11.5, fontweight="bold", va="top")
for i, (_, f) in enumerate(candidatas.iterrows()):
    lista.text(0, 0.88 - 0.088 * i, f"{f['upz'].title()} ({f['localidad'].title()})", fontsize=10, va="center")
    lista.text(0.95, 0.88 - 0.088 * i, f"{decimal(f['tasa_pct'], 1)} %", fontsize=10, va="center", ha="right")
titulo(fig, "Sin los cambios de plan, la pérdida de Hogar baja casi a la mitad y la del prepago no cambia",
       "Tasa mensual por segmento: de la S2 (cancelo, datos sin revisar) a la S3 (perdida, datos revisados); "
       "a la derecha, el 10 % de UPZ con más pérdida (al menos 80 clientes)", y=1.06)
fuente(fig, "Fuente: CONECTA S.A., datos sintéticos del curso (nov. 2025 – oct. 2026).")
guardar(fig, "que_cambia_s3.png")

## Errores frecuentes

1. **Limpiar todo lo que se ve raro.** Los 1 273 distritos con 52 años no afectan la razón precio/ingreso.
2. **No recalcular después de limpiar.** El umbral del 5 % cambia al quitar distritos.
3. **Imputar por costumbre.** Los nulos de una variable que el indicador no usa no se tocan.
4. **Limpiar sin bitácora.** Sin registro, nadie puede revisar ni repetir la decisión.
5. **Escribir a mano el mismo nombre en cada fila.** El nombre repetido miles de veces termina escrito de varias formas (en CONECTA, 38 formas de escribir 19 localidades).

## Síntesis

| Qué hicimos | Por qué importa |
|---|---|
| Corregimos la regla del Sankey de la S2 | Se revisan los datos y también las reglas con que se cuentan |
| Definimos la calidad como aptitud para el uso | Se limpia lo que la pregunta necesita |
| Encontramos valores límite e imposibles | 7 de las 10 mejores «gangas» eran problemas de datos |
| Revisamos los datos faltantes | Primero, si la pregunta usa la variable; después, de qué depende que falte. Los nulos de dormitorios no se tocan; el NPS no falta al azar |
| Registramos una bitácora | La lista pasa de 1 032 a 971 candidatos, con decisiones revisables |
| Guardamos la tabla revisada | Cada distrito con su condado y su región: de ahí parten las sesiones siguientes |

**Próxima sesión:** 971 candidatos para 150 cupos, casi todos en el interior. ¿Son oportunidades o solo lo
parecen? La S4 los verifica contra comparables y construye el gráfico principal.

## Transición al taller práctico

Cada equipo revisa la calidad de los datos de CONECTA S.A. (proyecto A: Retención) en
`sesion_03_practica_proyecto_A.ipynb`: encuentra las trampas, registra la bitácora, guarda la tabla limpia que
recibe la S4 y entrega las 10 UPZ candidatas.

## Referencias

Pace, R. K., y Barry, R. (1997). Sparse spatial autoregressions. *Statistics & Probability Letters, 33*(3), 291–297. https://doi.org/10.1016/S0167-7152(96)00140-X

U.S. Census Bureau. (2019). *2018 cartographic boundary file, current county and equivalent for United States, 1:20,000,000* [Shapefile].

van Buuren, S. (2018). *Flexible imputation of missing data* (2.ª ed.). CRC Press. https://stefvanbuuren.name/fimd/

Wang, R. Y., y Strong, D. M. (1996). Beyond accuracy: What data quality means to data consumers. *Journal of Management Information Systems, 12*(4), 5–33. https://doi.org/10.1080/07421222.1996.11518099

Wilke, C. O. (2019). *Fundamentals of data visualization: A primer on making informative and compelling figures*. O'Reilly Media.